# Swarm Pattern Example

This notebook demonstrates the Swarm pattern from Chapter 4.

## Overview
We'll build a game level design system where agents hand off control to each other:
- **Level Designer**: Creates level layouts, environments, and visual elements
- **Game Designer**: Playtests for fun and usability issues
- **Narrative Designer**: Ensures design supports the story and atmosphere
- **Difficulty Balancer**: Tests difficulty and pacing

Unlike supervisor-worker, the path isn't predetermined. Agents decide who should handle each issue as it comes up.

## Setup

In [ ]:
# Uncomment to install
# !pip install strands-agents

In [ ]:
from strands import Agent
from strands.multiagent import Swarm

In [ ]:
import logging
logging.getLogger("strands.multiagent").setLevel(logging.DEBUG)
logging.basicConfig(
    format="%(levelname)s | %(name)s | %(message)s",
    handlers=[logging.StreamHandler()]
)

## Create Specialist Agents

Each agent focuses on one aspect of game level design. They'll hand off to each other when they discover issues outside their expertise.

In [ ]:
level_designer = Agent(
    name="level_designer",
    model="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    system_prompt="""You are a Level Designer in a game design swarm.
Your role in the swarm is to create level layouts, environments, and visual elements.
Focus ONLY on layout and visual design. Do NOT write narrative or balance difficulty.
After completing your layout, hand off to narrative_designer for story elements 
and difficulty_balancer for gameplay tuning.
When receiving feedback from other agents, revise your design accordingly."""
)

game_designer = Agent(
    name="game_designer",
    model="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    system_prompt="""You are a Game Designer in a game design swarm.
Your role in the swarm is to playtest designs for fun, usability, and player confusion.
Focus ONLY on gameplay feel and player experience. Do NOT design layouts or write narrative.
When you find issues, hand off to the appropriate specialist to fix them.
Build upon the work of other agents while adding your playtesting perspective."""
)

narrative_designer = Agent(
    name="narrative_designer",
    model="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    system_prompt="""You are a Narrative Designer in a game design swarm.
Your role in the swarm is to create story elements, character backstories, and atmospheric details.
Focus ONLY on narrative. Do NOT redesign layouts or adjust difficulty.
After adding narrative elements, hand off to difficulty_balancer for balance review 
or game_designer for playtesting feedback.
When receiving designs from other agents, enrich them with story context."""
)

difficulty_balancer = Agent(
    name="difficulty_balancer",
    model="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    system_prompt="""You are a Difficulty Balancer in a game design swarm.
Your role in the swarm is to evaluate and tune difficulty progression, enemy placement, and pacing.
Focus ONLY on balance. Do NOT redesign layouts or write narrative.
When you find layout or narrative issues, hand off to the appropriate agent.
After balancing, hand off to game_designer for a final playtest review."""
)


## Create the Swarm

The Swarm automatically gives each agent a `handoff_to_agent` tool and shares conversation history across all agents.

Key parameters:
- `entry_point`: Which agent starts first
- `max_handoffs` / `max_iterations`: Prevent infinite loops
- `repetitive_handoff_detection_window` / `repetitive_handoff_min_unique_agents`: Prevent ping-pong between two agents

In [ ]:
swarm = Swarm(
    [level_designer, game_designer, narrative_designer, difficulty_balancer],
    entry_point=level_designer,
    max_handoffs=20,
    max_iterations=20,
    repetitive_handoff_detection_window=8,
    repetitive_handoff_min_unique_agents=3
)

print("✓ Swarm created")

## Run the Swarm

Watch how agents hand off to each other. The level designer starts, then other agents review and suggest changes. The path emerges from the conversation — it's not hardcoded.

In [ ]:
#result = swarm("Design a stealth mission level in an abandoned space station")
result = swarm("""Design a stealth mission level in an abandoned space station. 
The level needs to support multiple playstyles (aggressive stealth, patient stealth, and ghost runs). 
Include at least 3 distinct zones with different difficulty levels, 
narrative elements that explain why the station was abandoned, 
and ensure the difficulty curve is appropriate for mid-game players.""")

print("="*80)
print("FINAL RESULT:")
print("="*80)
print(result)

# See the handoff chain
print("\n" + "="*80)
print("HANDOFF CHAIN:")
print("="*80)
for node in result.node_history:
    print(f"  → {node.node_id}")

print(f"\nTotal iterations: {result.execution_count}")


## Key Takeaways

1. **Handoffs, not routing**: Agents pass control to each other explicitly — no central coordinator
2. **Shared history**: All agents see the full conversation, so they know what's been discussed
3. **Emergent workflow**: The sequence of agents isn't predetermined — it depends on what issues come up
4. **Safety limits**: `max_handoffs` and repetitive detection prevent infinite loops
5. **Use when**: The path can't be predicted upfront (creative work, iterative review, collaborative design)
6. **Don't use when**: You need a fixed sequence with guaranteed steps — use Graph pattern instead